# 02 - Noise EDA and Listening Samples

This notebook turns the noise/recovery definitions from `.agent_memory/noise_recovery_training_data.md` into inspectable tables and audio widgets.

Goal:
- identify the segments that are candidates for recovery;
- separate clean pathology targets from noisy/no-pathology noise snippets;
- listen to representative samples from each noise category;
- make simple synthetic clean+noise examples for denoising training intuition.

Important: the real dataset does **not** contain paired `(noisy, clean)` versions of the same segment. Real noisy segments are recovery/evaluation candidates. Supervised denoising pairs need to be synthesized from clean breathing segments plus extracted noise snippets.


In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, HTML, display

DATA_ROOT = Path('../data')
MR_DIR = DATA_ROOT / 'medical_records'
MR_WAVS = MR_DIR / 'ALL'
MR_JSON = MR_DIR / 'merged-1761225412230.json'

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 7


## 1. Load markup into a segment table

Every row below is one annotated interval from `files[*].markup[*]`. The source audio remains the original WAV; listening helpers slice by `start` and `length`.


In [ ]:
def has_value(x):
    return x is not None and x != '' and not (isinstance(x, float) and math.isnan(x))


WEAK_NOISE_QUALITIES = {'electric noise', 'quiet sound'}
STRONG_NOISE_QUALITIES = {
    'talk', 'crying', 'cough', 'movement', 'heart', 'other', 'unidentified', 'clothes'
}


def is_broad_noisy(row):
    quality = row.get('quality')
    noise_level = row.get('noise_level')
    return (has_value(quality) and quality != 'clean') or (has_value(noise_level) and noise_level != 'low')


def is_strong_noisy(row):
    quality = row.get('quality')
    noise_level = row.get('noise_level')
    return noise_level in {'moderate', 'high'} or quality in STRONG_NOISE_QUALITIES


def load_segment_dataframe(json_path=MR_JSON, wav_dir=MR_WAVS):
    with open(json_path, encoding='utf-8-sig') as f:
        payload = json.load(f)

    rows = []
    for file_idx, file_entry in enumerate(payload['files']):
        filename = file_entry['filename']
        wav_path = wav_dir / filename
        for markup_idx, segment in enumerate(file_entry.get('markup', [])):
            row = {
                'segment_id': f'{filename}:{markup_idx}',
                'file_idx': file_idx,
                'markup_idx': markup_idx,
                'uid': file_entry.get('uid'),
                'doctor': file_entry.get('doctor'),
                'filename': filename,
                'wav_path': str(wav_path),
                'wav_exists': wav_path.exists(),
                'start': float(segment.get('start', 0.0) or 0.0),
                'length': float(segment.get('length', 0.0) or 0.0),
                'phase': segment.get('phase'),
                'pathology': segment.get('pathology'),
                'quality': segment.get('quality'),
                'noise_level': segment.get('noise_level'),
                'breathe': segment.get('breathe'),
            }
            row['end'] = row['start'] + row['length']
            row['has_pathology'] = has_value(row['pathology'])
            row['broad_noisy'] = is_broad_noisy(row)
            row['strong_noisy'] = is_strong_noisy(row)
            row['clean_target'] = row['has_pathology'] and not row['broad_noisy']
            row['noise_only'] = (not row['has_pathology']) and row['broad_noisy']
            row['strong_noise_only'] = (not row['has_pathology']) and row['strong_noisy']
            row['recovery_candidate'] = row['strong_noisy']
            rows.append(row)

    df = pd.DataFrame(rows)
    df['duration_min'] = df['length'] / 60
    return df


segments = load_segment_dataframe()
segments.head()


## 2. High-level counts

These are the subsets we care about for denoising work:

- `clean_target`: clean pathology/breathing segments that can be the synthetic-pair target;
- `noise_only`: noisy segments without pathology labels, useful as noise donors;
- `strong_noise_only`: high-confidence noise donors;
- `recovery_candidate`: real-world noisy segments to run denoisers on and inspect/listen to.


In [ ]:
def subset_row(name, mask):
    part = segments.loc[mask]
    return {
        'subset': name,
        'segments': len(part),
        'duration_h': part['length'].sum() / 3600,
        'files': part['filename'].nunique(),
        'doctors': part['doctor'].nunique(),
        'median_segment_sec': part['length'].median() if len(part) else np.nan,
    }


summary = pd.DataFrame([
    subset_row('all annotated segments', segments.index == segments.index),
    subset_row('clean pathology targets', segments['clean_target']),
    subset_row('all broad noisy candidates', segments['broad_noisy']),
    subset_row('strong recovery candidates', segments['recovery_candidate']),
    subset_row('noisy with pathology labels', segments['has_pathology'] & segments['broad_noisy']),
    subset_row('noise-only snippets', segments['noise_only']),
    subset_row('strong noise-only snippets', segments['strong_noise_only']),
])

summary['duration_h'] = summary['duration_h'].round(3)
summary['median_segment_sec'] = summary['median_segment_sec'].round(3)
summary


In [ ]:
quality_counts = (
    segments['quality']
    .fillna('<missing>')
    .value_counts(dropna=False)
    .rename_axis('quality')
    .reset_index(name='segments')
)

noise_level_counts = (
    segments['noise_level']
    .fillna('<missing>')
    .replace('', '<empty>')
    .value_counts(dropna=False)
    .rename_axis('noise_level')
    .reset_index(name='segments')
)

display(HTML('<h4>quality</h4>'))
display(quality_counts)
display(HTML('<h4>noise_level</h4>'))
display(noise_level_counts)


In [ ]:
quality_noise_grid = pd.crosstab(
    segments['quality'].fillna('<missing>'),
    segments['noise_level'].fillna('<missing>').replace('', '<empty>'),
    margins=True,
)
quality_noise_grid


## 3. Candidate tables

Use these tables to inspect which recordings and interval labels will be used for training/evaluation.


In [ ]:
TABLE_COLS = [
    'segment_id', 'filename', 'doctor', 'start', 'length', 'phase',
    'pathology', 'quality', 'noise_level', 'breathe',
    'clean_target', 'noise_only', 'strong_noise_only', 'recovery_candidate',
]

clean_targets = segments.loc[segments['clean_target']].copy()
noise_only = segments.loc[segments['noise_only']].copy()
strong_noise_only = segments.loc[segments['strong_noise_only']].copy()
recovery_candidates = segments.loc[segments['recovery_candidate']].copy()

display(HTML('<h4>Clean pathology targets</h4>'))
display(clean_targets[TABLE_COLS].sample(min(20, len(clean_targets)), random_state=RANDOM_STATE).sort_values(['filename', 'start']))

display(HTML('<h4>Noise-only snippets</h4>'))
display(noise_only[TABLE_COLS].sample(min(20, len(noise_only)), random_state=RANDOM_STATE).sort_values(['quality', 'noise_level', 'filename', 'start']))

display(HTML('<h4>Strong recovery candidates</h4>'))
display(recovery_candidates[TABLE_COLS].sample(min(20, len(recovery_candidates)), random_state=RANDOM_STATE).sort_values(['quality', 'noise_level', 'filename', 'start']))


In [ ]:
category_summary = (
    segments.loc[segments['broad_noisy']]
    .assign(
        quality=segments['quality'].fillna('<missing>'),
        noise_level=segments['noise_level'].fillna('<missing>').replace('', '<empty>'),
    )
    .groupby(['quality', 'noise_level'], dropna=False)
    .agg(
        segments=('segment_id', 'count'),
        duration_min=('length', lambda s: s.sum() / 60),
        files=('filename', 'nunique'),
        with_pathology=('has_pathology', 'sum'),
        noise_only=('noise_only', 'sum'),
    )
    .reset_index()
    .sort_values(['segments', 'duration_min'], ascending=False)
)

category_summary['duration_min'] = category_summary['duration_min'].round(2)
category_summary


## 4. Audio helpers

The helpers below display a small table and an embedded audio player for each selected segment. Keep the sample counts low: notebook output gets large quickly.


In [ ]:
def read_segment(row, pad_sec=0.0, normalize=False):
    """Read one annotated segment from disk and return (audio, sample_rate)."""
    wav_path = Path(row['wav_path'])
    info = sf.info(str(wav_path))
    sr = info.samplerate
    start_sec = max(0.0, float(row['start']) - pad_sec)
    end_sec = min(float(info.frames) / sr, float(row['end']) + pad_sec)
    start_frame = int(round(start_sec * sr))
    frames = max(1, int(round((end_sec - start_sec) * sr)))
    audio, sr = sf.read(str(wav_path), start=start_frame, frames=frames, dtype='float32', always_2d=False)

    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if normalize:
        peak = np.max(np.abs(audio)) if len(audio) else 0.0
        if peak > 0:
            audio = audio / peak * 0.95
    return audio, sr


def compact_table(df):
    return df[TABLE_COLS].copy().reset_index(drop=True)


def listen_rows(df, n=3, random_state=RANDOM_STATE, title=None, pad_sec=0.05, normalize=True):
    if len(df) == 0:
        print('No rows to sample.')
        return pd.DataFrame(columns=TABLE_COLS)

    sample = df.sample(min(n, len(df)), random_state=random_state).sort_values(['filename', 'start'])
    if title:
        display(HTML(f'<h4>{title}</h4>'))
    display(compact_table(sample))

    for _, row in sample.iterrows():
        audio, sr = read_segment(row, pad_sec=pad_sec, normalize=normalize)
        label = (
            f"{row['segment_id']} | {row.get('quality')} | {row.get('noise_level')} | "
            f"pathology={row.get('pathology')} | {row['length']:.2f}s"
        )
        display(HTML(f'<b>{label}</b>'))
        display(Audio(audio, rate=sr))
    return sample


def plot_segment(row, pad_sec=0.05, normalize=True):
    audio, sr = read_segment(row, pad_sec=pad_sec, normalize=normalize)
    t = np.arange(len(audio)) / sr
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
    axes[0].plot(t, audio, linewidth=0.8)
    axes[0].set_title(f"Waveform: {row['segment_id']}")
    axes[0].set_xlabel('seconds')
    axes[0].set_ylabel('amplitude')

    axes[1].specgram(audio, Fs=sr, NFFT=512, noverlap=384, cmap='magma')
    axes[1].set_title('Spectrogram')
    axes[1].set_xlabel('seconds')
    axes[1].set_ylabel('Hz')
    axes[1].set_ylim(0, min(4000, sr // 2))
    plt.tight_layout()
    display(Audio(audio, rate=sr))
    return fig


## 5. Listen: clean targets vs broad recovery candidates


In [ ]:
_ = listen_rows(clean_targets, n=5, title='Clean pathology targets', random_state=1)


In [ ]:
_ = listen_rows(recovery_candidates, n=5, title='Strong recovery candidates', random_state=2)


## 6. Listen by noise quality

Change `N_PER_CATEGORY` or `QUALITY_ORDER` if the output becomes too large.


In [ ]:
N_PER_CATEGORY = 6
QUALITY_ORDER = [
    'electric noise', 'heart', 'talk', 'movement', 'cough', 'crying',
    'other', 'unidentified', 'quiet sound', 'clothes'
]

for quality in QUALITY_ORDER:
    part = segments.loc[(segments['quality'] == quality) & segments['broad_noisy']]
    if len(part) == 0:
        continue
    _ = listen_rows(
        part,
        n=N_PER_CATEGORY,
        random_state=RANDOM_STATE + 1,
        title=f'quality = {quality} ({len(part)} segments)',
    )


## 7. Listen by noise level

`moderate` and `high` are the most important for recovery/evaluation. `low` is useful to compare whether low electric noise is actually acceptable or still damaging.


In [ ]:
for level in ['low', 'moderate', 'high']:
    part = segments.loc[segments['noise_level'] == level]
    _ = listen_rows(
        part,
        n=4,
        random_state=RANDOM_STATE + 1,
        title=f'noise_level = {level} ({len(part)} segments)',
    )


## 8. Visual inspection: waveform and spectrogram

Pick any row from a candidate table and call `plot_segment(row)`. The example below samples one strong recovery candidate.


In [ ]:
example_row = recovery_candidates.sample(1).iloc[0]
display(compact_table(pd.DataFrame([example_row])))
_ = plot_segment(example_row)


## 9. Synthetic denoising-pair demo

This is the actual supervised-training idea: choose a clean pathology segment as the target, choose a noise-only snippet as interference, and mix them at a chosen SNR. The model input is `mixed`; the target is `clean`.

This cell is only for listening/intuition. A production dataset class should do this on the fly with random pairing, SNR sampling, resampling, padding/cropping, and train/val/test split hygiene.


In [ ]:
def rms(x):
    return float(np.sqrt(np.mean(np.square(x)))) if len(x) else 0.0


def crop_or_tile(x, n):
    if len(x) == n:
        return x
    if len(x) > n:
        return x[:n]
    if len(x) == 0:
        return np.zeros(n, dtype=np.float32)
    reps = int(np.ceil(n / len(x)))
    return np.tile(x, reps)[:n]


def resample_if_needed(audio, old_sr, new_sr):
    if old_sr == new_sr:
        return audio
    try:
        import librosa
        return librosa.resample(audio, orig_sr=old_sr, target_sr=new_sr)
    except Exception as exc:
        raise RuntimeError('Install librosa or choose same-sample-rate rows for this demo.') from exc


def mix_at_snr(clean, noise, snr_db):
    noise = crop_or_tile(noise, len(clean))
    clean_rms = rms(clean)
    noise_rms = rms(noise)
    if clean_rms == 0 or noise_rms == 0:
        return clean.copy(), noise
    target_noise_rms = clean_rms / (10 ** (snr_db / 20))
    scaled_noise = noise * (target_noise_rms / noise_rms)
    mixed = clean + scaled_noise
    peak = max(np.max(np.abs(clean)), np.max(np.abs(scaled_noise)), np.max(np.abs(mixed)), 1e-8)
    return mixed / peak * 0.95, scaled_noise / peak * 0.95


clean_row = clean_targets.sample(1, random_state=21).iloc[0]
noise_row = strong_noise_only.sample(1, random_state=22).iloc[0]
clean_audio, clean_sr = read_segment(clean_row, pad_sec=0.05, normalize=False)
noise_audio, noise_sr = read_segment(noise_row, pad_sec=0.05, normalize=False)
noise_audio = resample_if_needed(noise_audio, noise_sr, clean_sr)

SNR_DB = 0
mixed_audio, scaled_noise = mix_at_snr(clean_audio, noise_audio, SNR_DB)

clean_listen = clean_audio / max(np.max(np.abs(clean_audio)), 1e-8) * 0.95

display(HTML('<h4>Clean target</h4>'))
display(compact_table(pd.DataFrame([clean_row])))
display(Audio(clean_listen, rate=clean_sr))

display(HTML('<h4>Noise donor</h4>'))
display(compact_table(pd.DataFrame([noise_row])))
display(Audio(scaled_noise, rate=clean_sr))

display(HTML(f'<h4>Synthetic noisy input at {SNR_DB} dB SNR</h4>'))
display(Audio(mixed_audio, rate=clean_sr))


## 10. Export candidate tables if needed

Uncomment this cell if we want CSV artifacts for later scripts. It is commented by default to keep notebook execution read-only.


In [ ]:
# OUT_DIR = Path('../data/noise_eda_tables')
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# clean_targets.to_csv(OUT_DIR / 'clean_targets.csv', index=False)
# noise_only.to_csv(OUT_DIR / 'noise_only.csv', index=False)
# strong_noise_only.to_csv(OUT_DIR / 'strong_noise_only.csv', index=False)
# recovery_candidates.to_csv(OUT_DIR / 'recovery_candidates.csv', index=False)
